In [17]:
import pandas as pd
import glob
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import time



# CSV-Daten laden
keywords = ["germany", "german", "deutschland", "nato", "bundeswehr", "scholz", "leopard"]
dataset_path = glob.glob("C:\\Users\\Benedikt Thissen\\OneDrive\\Dokumente\\1Studium\\Data Analysis\\Datenset\\*.csv")
corpus = []
for data in tqdm(dataset_path, desc="Lade und filtere CSV-Dateien"):
    df = pd.read_csv(data)
    df = df[df["content"].notna()]  
    filtered_df = df[df["content"].str.lower().str.contains("|".join(keywords), na=False)]
    corpus.extend(filtered_df["content"].tolist())


#Textbereinigung
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


# Entfernt Wörter mit mehr als 4 Wiederholungen desselben Zeichens
def is_valid_word(word):
    return not re.search(r"(.)\1{4,}", word)

sentences = []
for text in tqdm(corpus, desc="Bereinige Texte"):
    tokens = re.findall(r'\b\w+\b', text.lower())  # einfache Tokenisierung ohne Satzzeichen
    cleaned = [lemmatizer.lemmatize(w) for w in tokens 
               if w.isalpha() and w not in stop_words and is_valid_word(w)]
    sentences.append(cleaned)
 
#BoW-Matrix erstellen
joined_sentences = [" ".join(sentence) for sentence in sentences]
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(tqdm(joined_sentences, desc="BoW Verarbeitung"))
print(f"BoW-Matrix erstellt: {bow_matrix.shape}")

# LDA-Modell
n_topics = 5
print("Starte LDA-Training...") #Benachrichtigung über Start
start_time = time.time() #LDA Timer
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_model.fit(bow_matrix)
end_time = time.time()
print(f"LDA-Training abgeschlossen in {end_time - start_time:.2f} Sekunden.")

# Top-Wörter pro Thema anzeigen
feature_names = vectorizer.get_feature_names_out()
for idx, topic in enumerate(lda_model.components_):
    print(f"\nThema {idx + 1}:")
    top_words = topic.argsort()[-20:][::-1]
    print(", ".join([feature_names[i] for i in top_words]))



BoW Verarbeitung: 100%|██████████| 398364/398364 [00:06<00:00, 65315.07it/s]


BoW-Matrix erstellt: (398364, 218994)
Starte LDA-Training...
LDA-Training abgeschlossen in 1146.13 Sekunden.

Thema 1:
co, http, ukraine, nato, russia, amp, war, putin, russian, crisis, via, eu, standwithukraine, usa, support, troop, say, ukrainian, uk, biden

Thema 2:
die, nato, ukraine, der, und, nicht, da, russland, ist, co, http, zu, putin, mit, hat, den, von, ein, sich, auch

Thema 3:
ukraine, nato, russia, troop, co, http, russian, border, germany, military, europe, force, putin, say, eastern, biden, ally, invasion, poland, near

Thema 4:
nato, ukraine, russia, putin, war, would, want, country, russian, border, invade, join, troop, amp, like, think, member, get, right, one

Thema 5:
nato, ukraine, de, rusembusa, bbcworld, russianembassy, la, en, rusembukraine, statedept, er, russiaun, et, det, og, le, ikke, skynews, co, http


Analyse der Themen. 
Thema 1: Unterstützung der USA für die Ukraine
Thema 2: Hier werden viele deutsche Wörter verwendet. Inhaltlich genauer anschauen
Thema 3: Europäische Militärpräsenz
Thema 4: Begründung des Krieges
Thema 5: Internationale Berichterstattung
